# Silver DQ: §6.1.2 metrics runner

Computes the four §6.1.2 families (completeness · consistency · uniqueness · timeliness)
against a silver Iceberg table, overall **and** per-MNO, and writes `summary.json` +
`metrics.json` to `poc/dq/results/dataset=<name>/run_date=<YYYY-MM-DD>/`.

**How to run:** run the engine cells once, set the run options, then run **exactly one**
of the four table cells: that picks the table and its `checks.yaml` together. Everything
below is table-agnostic; it all comes from the yaml.

⚠️ **Not a Run All notebook.** Run All executes all four table cells and the last one
(`tu_portps`) wins. Pick your table, then run from *Read the silver table* down.

In [1]:
%%pyspark project.spark.compatibility
import json, statistics
from datetime import date, timedelta
from pyspark.sql import SparkSession, DataFrame, Row
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("silver_dq_run")
    .config("spark.executor.memory", "18g")
    .config("spark.driver.memory", "10g")
    .config("spark.executor.cores", "4")
    .config("spark.sql.shuffle.partitions", "300")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    # S3FileIO (AWS SDK v2) HTTP pool. Default is small; a full-table scan at
    # executor.cores=4 across ~10 executors exhausts it and every task dies with
    # "Timeout waiting for connection from pool". Applied ONLY when the session is
    # CREATED restart the kernel for these to take effect. Catalog name here is
    # spark_catalog; adjust if your Iceberg catalog is registered under another name.
    .config("spark.sql.catalog.spark_catalog.http-client.type", "apache")
    .config("spark.sql.catalog.spark_catalog.http-client.apache.max-connections", "500")
    .config("spark.sql.catalog.spark_catalog.http-client.apache.connection-acquisition-timeout-ms", "60000")
    .config("spark.sql.catalog.spark_catalog.http-client.apache.connection-timeout-ms", "60000")
    .config("spark.sql.catalog.spark_catalog.http-client.apache.socket-timeout-ms", "120000")
    .getOrCreate()
)


Executing for connection type: SPARK_GLUE, connection name: project.spark.compatibility
Creating Glue session...
Create session for connection: project.spark.compatibility


'Session ae2xwmv6edkt56-4ed3ebeb-1b86-408e-b13b-b810645c46fa has been created.'

<sagemaker_studio_dataengineering_sessions.sagemaker_base_session_manager.common.debugging_utils.SessionInfoTableDisplay object>

Session created for connection: project.spark.compatibility.

Connection: project.spark.compatibility | Run start time: 2026-07-22 00:46:37.693023 | Run duration : 0:02:26.940707s.


### Engine: parse checks.yaml into a CheckConfig

One dataclass carrying that table's whole DQ contract: which columns must be
non-null, what the primary key is, which column splits the per-MNO breakdown,
each column's allowed value set, and how timeliness is measured.

In [2]:
%%pyspark project.spark.compatibility
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional


@dataclass(frozen=True)
class CheckConfig:
    """One silver table's DQ contract, parsed from that table's checks.yaml."""
    dataset: str
    primary_keys: list = field(default_factory=list)        # [] -> full-row
    partition_by: Optional[str] = None
    event_timestamp_col: str = "event_timestamp"

    timeliness_enabled: bool = False                        # set by presence of `timeliness:` block
    timeliness_mode: str = "batch"                          # batch = event-time | streaming = ingestion gap
    future_grace_seconds: int = 0                           # batch mode
    max_recency_seconds: Optional[int] = None               # batch mode, optional gate
    ingestion_timestamp_col: str = "ingestion_ts"           # streaming mode
    max_delay_seconds: Optional[int] = None                 # streaming mode, optional gate

    required_columns: list = field(default_factory=list)
    allowed_values: dict = field(default_factory=dict)              # {col:[vals]}
    allowed_values_by_mno: dict = field(default_factory=dict)       # {col:{mno:[vals]}}
    validity_formats: dict = field(default_factory=dict)            # {col: java-regex} (S12 Validity)
    victim_exclusion_col: Optional[str] = None                      # S12 Victim exclusion guard
    victim_allowed_values: list = field(default_factory=list)       #   values that mean "confirmed bad actor"
    currency_timestamp_col: str = "ingestion_ts"                    # S12 Currency: per-row age basis
    currency_max_age_seconds: Optional[int] = None                  #   review cycle; governance-provided

    def __post_init__(self):
        if not self.dataset:
            raise ValueError("checks.yaml must set 'dataset'")
        if self.allowed_values_by_mno and self.partition_by is None:
            raise ValueError("allowed_values_by_mno set but 'partition_by' (MNO col) is not")
        if self.timeliness_mode not in ("batch", "streaming"):
            raise ValueError(f"timeliness.mode must be 'batch' or 'streaming', got {self.timeliness_mode!r}")
        if (self.victim_exclusion_col is None) != (not self.victim_allowed_values):
            raise ValueError("victim_exclusion needs both 'column' and a non-empty 'allowed_values'")
        for col, pattern in (self.validity_formats or {}).items():
            if not isinstance(pattern, str) or not pattern:
                raise ValueError(f"validity.formats.{col} must be a non-empty regex string")
        if self.currency_max_age_seconds is not None and int(self.currency_max_age_seconds) <= 0:
            raise ValueError("timeliness.currency.max_age_seconds must be positive")


def parse_checks(raw: dict) -> CheckConfig:
    c = raw or {}
    completeness = c.get("completeness") or {}
    consistency  = c.get("consistency") or {}
    validity     = c.get("validity") or {}
    victim       = c.get("victim_exclusion") or {}
    tl           = c.get("timeliness") or {}
    currency     = tl.get("currency") or {}
    return CheckConfig(
        dataset=c.get("dataset"),
        primary_keys=list(c.get("primary_keys") or []),
        partition_by=c.get("partition_by"),
        event_timestamp_col=c.get("event_timestamp_col", "event_timestamp"),
        timeliness_enabled=("timeliness" in c),
        timeliness_mode=tl.get("mode", "batch"),
        future_grace_seconds=int(tl.get("future_grace_seconds", 0)),
        max_recency_seconds=tl.get("max_recency_seconds"),
        ingestion_timestamp_col=tl.get("ingestion_timestamp_col", "ingestion_ts"),
        max_delay_seconds=tl.get("max_delay_seconds"),
        required_columns=list(completeness.get("required_columns") or []),
        allowed_values=dict(consistency.get("allowed_values") or {}),
        allowed_values_by_mno=dict(consistency.get("allowed_values_by_mno") or {}),
        validity_formats=dict(validity.get("formats") or {}),
        victim_exclusion_col=victim.get("column"),
        victim_allowed_values=list(victim.get("allowed_values") or []),
        currency_timestamp_col=currency.get("timestamp_col", "ingestion_ts"),
        currency_max_age_seconds=currency.get("max_age_seconds"),
    )


def load_checks(path: str) -> CheckConfig:
    """Read checks.yaml from S3."""
    import boto3, yaml
    if not path.startswith("s3://"):
        raise ValueError(f"CHECKS_PATH must be an s3:// uri, got: {path}")
    bucket, key = path[len("s3://"):].split("/", 1)
    return parse_checks(yaml.safe_load(boto3.client("s3").get_object(Bucket=bucket, Key=key)["Body"].read()))


Connection: project.spark.compatibility | Run start time: 2026-07-22 00:49:04.639513 | Run duration : 0:00:01.215538s.


### Engine: the metric families, as Spark aggregations + parsers

Each family contributes two halves: `*_exprs` returns aggregation columns that
count offending rows, and `parse_*` turns those raw counts into rates and
pass/fail verdicts. Split this way so every family can share one scan.

In [3]:
%%pyspark project.spark.compatibility
from pyspark.sql import Column
from pyspark.sql import functions as F

_NULL_SENTINEL = "\u0000NULL\u0000"
_KEY_SEP = "\u0001"


def alias_safe_name(column_name: str) -> str:
    """Strip characters that can't appear in an aggregation alias."""
    return column_name.replace(".", "_").replace("`", "")


def resolve_column(name, columns):
    """The table column matching `name` case-insensitively, or None.

    Same identifier rule Spark itself uses. Every family that takes a column name
    from the yaml resolves through here; see parse_completeness for the live
    incident that made case-blind matching mandatory.
    """
    fold = name.casefold()
    for c in columns:
        if c.casefold() == fold:
            return c
    return None


# completeness #
def completeness_exprs(columns):
    return [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"null_rows_{alias_safe_name(c)}")
            for c in columns]


def parse_completeness(row, columns, row_count, required):
    per_column, rules = {}, []
    # Yaml names must match table names the way Spark resolves columns: case-
    # insensitively. Proven live 2026-07-17 on account_changes_batch: Iceberg
    # preserves column case, the table has phone_number_AC_hash, the yaml says
    # phone_number_ac_hash a case-sensitive `c in required` produced 6 rules
    # from 7 required columns and the missing gate left no trace.
    required_by_fold = {r.casefold(): r for r in sorted(required)}
    matched_folds = set()
    for c in columns:
        nulls = int(row[f"null_rows_{alias_safe_name(c)}"] or 0)
        rate = round(nulls / row_count, 6) if row_count else 0.0
        per_column[c] = {"null_count": nulls, "null_rate": rate}
        if c.casefold() in required_by_fold:
            matched_folds.add(c.casefold())
            rules.append({"family": "completeness", "rule": f"{c} is not null",
                          "column": c, "violations": nulls, "passed": nulls == 0})
    # A required column the table lacks under ANY casing is a failed rule, not a
    # silent no-op otherwise a typo or a renamed column turns its gate off.
    for fold in sorted(set(required_by_fold) - matched_folds):
        name = required_by_fold[fold]
        rules.append({"family": "completeness", "rule": f"{name} exists in the table",
                      "column": name, "violations": 1, "passed": False})
    return {"per_column": per_column, "rules": rules}


# consistency #
def consistency_exprs(allowed_values, allowed_values_by_mno, mno_col):
    exprs = []
    for col, allowed in allowed_values.items():
        viol = F.col(col).isNotNull() & ~F.col(col).isin(list(allowed))
        exprs.append(F.sum(F.when(viol, 1).otherwise(0))
                     .alias(f"domain_violation_rows_{alias_safe_name(col)}"))
    for col, by_mno in allowed_values_by_mno.items():
        cond = F.lit(False)
        for mno, allowed in by_mno.items():
            cond = cond | ((F.col(mno_col) == mno) & F.col(col).isNotNull()
                           & ~F.col(col).isin(list(allowed)))
        exprs.append(F.sum(F.when(cond, 1).otherwise(0))
                     .alias(f"mno_domain_violation_rows_{alias_safe_name(col)}"))
    return exprs


def parse_consistency(row, allowed_values, allowed_values_by_mno, row_count):
    per_column, rules = {}, []
    for col, allowed in allowed_values.items():
        v = int(row[f"domain_violation_rows_{alias_safe_name(col)}"] or 0)
        per_column[col] = {"violation_count": v,
                           "violation_rate": round(v / row_count, 6) if row_count else 0.0,
                           "allowed_values": list(allowed)}
        rules.append({"family": "consistency", "rule": f"{col} in {sorted(map(str, allowed))}",
                      "column": col, "violations": v, "passed": v == 0})
    for col, by_mno in allowed_values_by_mno.items():
        v = int(row[f"mno_domain_violation_rows_{alias_safe_name(col)}"] or 0)
        per_column[col] = {"violation_count": v,
                           "violation_rate": round(v / row_count, 6) if row_count else 0.0,
                           "allowed_values_by_mno": {m: list(a) for m, a in by_mno.items()}}
        rules.append({"family": "consistency", "rule": f"{col} in per-MNO allowed set",
                      "column": col, "violations": v, "passed": v == 0})
    return {"per_column": per_column, "rules": rules}


# validity #
def validity_exprs(formats, columns):
    """One mismatch counter per configured format whose column exists (any casing).

    NULLs are completeness's job; a format check counts only non-null values that
    fail the pattern. Patterns are Java regex (Spark rlike) matched ANYWHERE in
    the value; anchor with ^...$ to require a full-string match.
    """
    exprs = []
    for name, pattern in sorted(formats.items()):
        c = resolve_column(name, columns)
        if c is not None:
            exprs.append(
                F.sum(F.when(F.col(c).isNotNull() & ~F.col(c).rlike(pattern), 1).otherwise(0))
                 .alias(f"format_violation_rows_{alias_safe_name(c)}"))
    return exprs


def parse_validity(row, formats, columns, row_count):
    per_column, rules = {}, []
    for name, pattern in sorted(formats.items()):
        c = resolve_column(name, columns)
        if c is None:
            # Fail closed, same as completeness: a format on a column the table
            # lacks under any casing must go red, not silently vanish.
            rules.append({"family": "validity", "rule": f"{name} exists in the table",
                          "column": name, "violations": 1, "passed": False})
            continue
        bad = int(row[f"format_violation_rows_{alias_safe_name(c)}"] or 0)
        per_column[c] = {"pattern": pattern, "violation_count": bad,
                         "violation_rate": round(bad / row_count, 6) if row_count else 0.0}
        rules.append({"family": "validity", "rule": f"{c} matches format",
                      "column": c, "violations": bad, "passed": bad == 0})
    return {"per_column": per_column, "rules": rules}


# uniqueness (null-safe key) #
def uniqueness_exprs(primary_keys, all_columns):
    keys = primary_keys or all_columns
    # coalesce each key part to a sentinel BEFORE concat (concat_ws drops nulls),
    # hash to fixed length, count distinct. A raw countDistinct over the key columns
    # would skip any row with a null key part -> wrong dedup rate on tables with
    # nullable keys (e.g. audit_trail full-row dedup where msisdn is 100% null).
    key = F.sha2(F.concat_ws(
        _KEY_SEP,
        *[F.coalesce(F.col(k).cast("string"), F.lit(_NULL_SENTINEL)) for k in keys],
    ), 256)
    return [F.countDistinct(key).alias("distinct_key_rows")]


def parse_uniqueness(row, primary_keys, row_count):
    distinct = int(row["distinct_key_rows"] or 0)
    dupes = row_count - distinct
    keys = primary_keys or ["<full row>"]
    return {"metrics": {"primary_keys": keys, "distinct_keys": distinct,
                        "duplicate_count": dupes,
                        "duplicate_rate": round(dupes / row_count, 6) if row_count else 0.0,
                        "distinct_rate": round(distinct / row_count, 6) if row_count else 0.0},
            "rules": [{"family": "uniqueness", "rule": f"unique on {keys}",
                       "column": ",".join(keys), "violations": dupes, "passed": dupes == 0}]}


# timeliness #
# Two modes, chosen by `timeliness.mode` in the yaml:
#
# batch event-time only. Gate = no event dated after the run. Use while the
# table is a one-shot historical backfill: every row's ingestion_ts is
# ~the same instant, so an ingestion gap measures "how old is this
# event" (0-3 years), not "how slow is the pipeline" no threshold
# is correct. Today all 4 tables are here, and 3 of them write
# ingestion_ts as an F.lit placeholder anyway.
#
# streaming ingestion gap = ingestion_ts - event_timestamp, the AC's definition. Gate =
# no row later than max_delay_seconds. Only meaningful once the table
# is fed incrementally AND ingestion_ts is stamped at write time.
#
# Both emit min/max_event_time so coverage is comparable across modes.
def timeliness_exprs(cfg, run_date, columns):
    ev = F.col(cfg.event_timestamp_col)
    exprs = [
        F.min(ev).cast("string").alias("min_event_time"),
        F.max(ev).cast("string").alias("max_event_time"),
        F.sum(F.when(ev.isNull(), 1).otherwise(0)).alias("null_event_time_rows"),
    ]
    if cfg.timeliness_mode == "streaming":
        ing = F.col(cfg.ingestion_timestamp_col)
        gap = F.unix_timestamp(ing) - F.unix_timestamp(ev)
        exprs += [
            F.min(ing).cast("string").alias("min_ingestion_time"),
            F.max(ing).cast("string").alias("max_ingestion_time"),
            F.max(gap).alias("max_gap_seconds"),
            F.sum(F.when(gap < 0, 1).otherwise(0)).alias("negative_gap_rows"),
            F.sum(F.when(ing.isNull(), 1).otherwise(0)).alias("null_ingestion_time_rows"),
            F.sum(F.when(F.lit(cfg.max_delay_seconds is not None)
                         & (gap > F.lit(cfg.max_delay_seconds or 0)), 1).otherwise(0))
             .alias("delay_violation_rows"),
        ]
    else:
        ev_unix = F.unix_timestamp(ev)
        as_of_unix = F.unix_timestamp(F.date_add(F.to_date(F.lit(run_date)), 1))  # midnight after run_date
        future_cutoff = as_of_unix + F.lit(int(cfg.future_grace_seconds))
        exprs += [
            F.min(as_of_unix - ev_unix).alias("recency_seconds"),   # == as_of - max(event_time)
            F.sum(F.when(ev_unix > future_cutoff, 1).otherwise(0)).alias("future_event_rows"),
        ]
    # S12 Currency: per-row review-cycle gate, valid in either mode. Age basis
    # defaults to ingestion_ts; as-of = midnight after run_date, the same instant
    # batch recency uses.
    if cfg.currency_max_age_seconds is not None:
        cur = resolve_column(cfg.currency_timestamp_col, columns)
        if cur is not None:
            cur_as_of = F.unix_timestamp(F.date_add(F.to_date(F.lit(run_date)), 1))
            age = cur_as_of - F.unix_timestamp(F.col(cur))
            exprs.append(
                F.sum(F.when(age > F.lit(int(cfg.currency_max_age_seconds)), 1).otherwise(0))
                 .alias("stale_rows"))
    return exprs


def parse_timeliness(row, cfg, columns, row_count):
    _rate = lambda n: round(n / row_count, 6) if row_count else 0.0
    metrics = {
        "mode": cfg.timeliness_mode,
        "event_timestamp_col": cfg.event_timestamp_col,
        "min_event_time": row["min_event_time"],
        "max_event_time": row["max_event_time"],
        "null_event_time_count": int(row["null_event_time_rows"] or 0),
    }
    if cfg.timeliness_mode == "streaming":
        delayed = int(row["delay_violation_rows"] or 0)
        negative = int(row["negative_gap_rows"] or 0)
        metrics.update({
            "ingestion_timestamp_col": cfg.ingestion_timestamp_col,
            "min_ingestion_time": row["min_ingestion_time"],
            "max_ingestion_time": row["max_ingestion_time"],
            "max_gap_seconds": int(row["max_gap_seconds"]) if row["max_gap_seconds"] is not None else None,
            "negative_gap_count": negative,
            "negative_gap_rate": _rate(negative),
            "null_ingestion_time_count": int(row["null_ingestion_time_rows"] or 0),
            "delay_violation_count": delayed,
            "delay_violation_rate": _rate(delayed),
            "max_delay_seconds": cfg.max_delay_seconds,
        })
        rules = [{"family": "timeliness", "rule": "no event_time after ingestion_time",
                  "column": cfg.event_timestamp_col, "violations": negative, "passed": negative == 0}]
        if cfg.max_delay_seconds is not None:
            rules.append({"family": "timeliness",
                          "rule": f"ingestion gap <= {cfg.max_delay_seconds}s",
                          "column": cfg.event_timestamp_col,
                          "violations": delayed, "passed": delayed == 0})
    else:
        future = int(row["future_event_rows"] or 0)
        recency = int(row["recency_seconds"]) if row["recency_seconds"] is not None else None
        metrics.update({
            "recency_seconds": recency,
            "future_count": future,
            "future_rate": _rate(future),
            "max_recency_seconds": cfg.max_recency_seconds,
        })
        rules = [{"family": "timeliness", "rule": "no future event_time",
                  "column": cfg.event_timestamp_col, "violations": future, "passed": future == 0}]
        if cfg.max_recency_seconds is not None:
            over = 1 if (recency is not None and recency > cfg.max_recency_seconds) else 0
            rules.append({"family": "timeliness", "rule": f"recency <= {cfg.max_recency_seconds}s",
                          "column": cfg.event_timestamp_col, "violations": over, "passed": over == 0})
    if cfg.currency_max_age_seconds is not None:
        cur = resolve_column(cfg.currency_timestamp_col, columns)
        if cur is None:
            rules.append({"family": "timeliness",
                          "rule": f"{cfg.currency_timestamp_col} exists in the table",
                          "column": cfg.currency_timestamp_col, "violations": 1, "passed": False})
        else:
            stale = int(row["stale_rows"] or 0)
            metrics["currency"] = {"timestamp_col": cur,
                                   "max_age_seconds": int(cfg.currency_max_age_seconds),
                                   "stale_count": stale, "stale_rate": _rate(stale)}
            rules.append({"family": "timeliness",
                          "rule": f"rows within review cycle ({cur} age <= {int(cfg.currency_max_age_seconds)}s)",
                          "column": cur, "violations": stale, "passed": stale == 0})
    return {"metrics": metrics, "rules": rules}


# victim exclusion #
def victim_exclusion_exprs(col_name, allowed, columns):
    """One counter: rows NOT attested as confirmed bad actor.

    NULL is a violation here, unlike consistency: an unattested record must not
    sit in the matchable population (governance framework S7.3 makes both
    attestations mandatory at ingestion), so a missing value fails the same as a
    wrong one.
    """
    c = resolve_column(col_name, columns)
    if c is None:
        return []
    return [F.sum(F.when(F.col(c).isin(list(allowed)), 0).otherwise(1))
             .alias("victim_violation_rows")]


def parse_victim_exclusion(row, col_name, allowed, columns, row_count):
    c = resolve_column(col_name, columns)
    if c is None:
        # Fail closed: a guard pointed at a column the table lacks is OFF, and an
        # off guard must be loud (see parse_completeness for the live incident).
        return {"metrics": {"column": col_name, "allowed_values": sorted(allowed),
                            "violation_count": None, "violation_rate": None},
                "rules": [{"family": "victim_exclusion",
                           "rule": f"{col_name} exists in the table",
                           "column": col_name, "violations": 1, "passed": False}]}
    bad = int(row["victim_violation_rows"] or 0)
    return {"metrics": {"column": c, "allowed_values": sorted(allowed),
                        "violation_count": bad,
                        "violation_rate": round(bad / row_count, 6) if row_count else 0.0},
            "rules": [{"family": "victim_exclusion",
                       "rule": f"{c} attests confirmed bad actor (nulls fail)",
                       "column": c, "violations": bad, "passed": bad == 0}]}


Connection: project.spark.compatibility | Run start time: 2026-07-22 00:49:05.862739 | Run duration : 0:00:00.769666s.


### Engine: scan the table once, then shape summary.json and metrics.json

`_scan` is the only Spark step: it fires every family's aggregations in a
single pass (plus one more if there's a per-MNO breakdown). `build_summary`
and `build_metrics` are pure Python: they reshape that one result into the two
output files, so the files can't disagree.

In [4]:
%%pyspark project.spark.compatibility
SCHEMA_VERSION = "1.0.0"
FAMILIES = ("completeness", "consistency", "validity", "uniqueness", "timeliness", "victim_exclusion")


def _timeliness_on(cfg, columns):
    """Timeliness runs only if the yaml asked for it AND the columns it needs exist."""
    if not cfg.timeliness_enabled or cfg.event_timestamp_col not in columns:
        return False
    if cfg.timeliness_mode == "streaming":
        return cfg.ingestion_timestamp_col in columns
    return True


def _scan(df, cfg, columns, with_partition, run_date):
    """Read the table and return the raw per-family counts. THE ONLY SPARK STEP.

    Every family is concatenated into ONE .agg(), so a DQ run costs at most
    two passes over the table (one overall + one grouped by MNO) no matter how
    many rules the yaml configures.

    Do NOT split this per family or per output file. account_changes_batch is
    ~400M rows and audit_trail_services_3 ~436M; every extra split is another
    full pass. It would also let summary.json and metrics.json disagree: they are
    consistent today precisely because both are shaped from this one result.
    """
    exprs = [F.count(F.lit(1)).alias("row_count")]
    exprs += completeness_exprs(columns)
    exprs += consistency_exprs(cfg.allowed_values, cfg.allowed_values_by_mno, cfg.partition_by)
    exprs += validity_exprs(cfg.validity_formats, columns)
    exprs += uniqueness_exprs(cfg.primary_keys, columns)
    if cfg.victim_exclusion_col:
        exprs += victim_exclusion_exprs(cfg.victim_exclusion_col, cfg.victim_allowed_values, columns)
    if _timeliness_on(cfg, columns):
        exprs += timeliness_exprs(cfg, run_date, columns)
    if with_partition:
        return df.groupBy(cfg.partition_by).agg(*exprs).collect()
    return df.agg(*exprs).collect()


def _families_from_row(row, cfg, columns):
    """One _scan Row -> {row_count, <family>: {numbers, rules}}. Rates and verdicts happen here."""
    row_count = int(row["row_count"] or 0)
    out = {
        "row_count": row_count,
        "completeness": parse_completeness(row, columns, row_count, set(cfg.required_columns)),
        "consistency": parse_consistency(row, cfg.allowed_values, cfg.allowed_values_by_mno, row_count),
        "uniqueness": parse_uniqueness(row, cfg.primary_keys, row_count),
    }
    if cfg.validity_formats:
        out["validity"] = parse_validity(row, cfg.validity_formats, columns, row_count)
    if cfg.victim_exclusion_col:
        out["victim_exclusion"] = parse_victim_exclusion(
            row, cfg.victim_exclusion_col, cfg.victim_allowed_values, columns, row_count)
    if _timeliness_on(cfg, columns):
        out["timeliness"] = parse_timeliness(row, cfg, columns, row_count)
    return out


def _metrics_view(families):
    """Keep the numbers, drop the verdicts (verdicts are summary.json's job)."""
    out = {"row_count": families["row_count"]}
    for fam in FAMILIES:
        if fam in families:
            out[fam] = {k: v for k, v in families[fam].items() if k in ("per_column", "metrics")}
    return out


def build_summary(overall, cfg, run_date):
    """The verdicts -> summary.json. Pure Python: feed it a _families_from_row dict, no Spark."""
    rules = []
    for fam in FAMILIES:
        if fam in overall:
            rules.extend(overall[fam]["rules"])

    # An empty table satisfies every "violations == 0" rule vacuously: 0 nulls, 0 domain
    # violations, 0 duplicates, 0 future events -> passed=true for a table with no data.
    # That is the exact false green this module exists to prevent (see the ticket's
    # Motivation: lineage needs a dependable "is this silver run trustworthy" signal).
    # Verified 2026-07-16 on a real run: device_lookup_batch at row_count=0 reported
    # 16/16 rules passed and validated cleanly against the committed schema.
    if overall["row_count"] == 0:
        rules.append({"family": "completeness", "rule": "table is not empty",
                      "column": "*", "violations": 1, "passed": False})

    passed = sum(1 for r in rules if r["passed"])
    return {
        "schema_version": SCHEMA_VERSION,
        "dataset": cfg.dataset,
        "run_date": run_date,
        "row_count": overall["row_count"],
        "rule_count": len(rules),
        "passed_count": passed,
        "failed_count": len(rules) - passed,
        "passed": all(r["passed"] for r in rules),
        "rules": sorted(rules, key=lambda r: (r["family"], r["column"], r["rule"])),
    }


def build_metrics(overall, per_mno, cfg, run_date):
    """The numbers -> metrics.json. Pure Python: feed it _families_from_row dicts, no Spark."""
    return {
        "schema_version": SCHEMA_VERSION,
        "dataset": cfg.dataset,
        "run_date": run_date,
        "partition_by": cfg.partition_by,
        "overall": _metrics_view(overall),
        "per_mno": {k: _metrics_view(v) for k, v in per_mno.items()},
    }


def compute_dq(df, cfg, run_date):
    """Scan once, shape twice. Deterministic: reads only the DataFrame + cfg, never wall-clock."""
    columns = list(df.columns)
    overall = _families_from_row(_scan(df, cfg, columns, False, run_date)[0], cfg, columns)
    per_mno = {}
    if cfg.partition_by is not None:
        for r in _scan(df, cfg, columns, True, run_date):
            key = r[cfg.partition_by]
            per_mno[str(key) if key is not None else "__null__"] = _families_from_row(r, cfg, columns)
    return (build_summary(overall, cfg, run_date),
            build_metrics(overall, per_mno, cfg, run_date))


Connection: project.spark.compatibility | Run start time: 2026-07-22 00:49:06.637854 | Run duration : 0:00:01.234668s.


### Engine: result paths, JSON serialization, S3 write

`_dumps` is defined here rather than next to the write cell because the Inspect
cells print through it: what you read below is exactly what lands in S3.

In [5]:
%%pyspark project.spark.compatibility
import json, os


def result_paths(dq_root, cfg, run_date):
    base = f"{dq_root.rstrip('/')}/results/dataset={cfg.dataset}/run_date={run_date}"
    return f"{base}/summary.json", f"{base}/metrics.json"


def _dumps(obj):
    """The one serializer. The Inspect cells print through this too, so what you read
    in the notebook is byte-for-byte what write_results puts in S3."""
    return json.dumps(obj, sort_keys=True, indent=2, ensure_ascii=False)


def write_results(summary, metrics, dq_root, cfg, run_date):
    s_path, m_path = result_paths(dq_root, cfg, run_date)
    for path, obj in ((s_path, summary), (m_path, metrics)):
        payload = _dumps(obj)
        if path.startswith("s3://"):
            import boto3
            bucket, key = path[len("s3://"):].split("/", 1)
            boto3.client("s3").put_object(Bucket=bucket, Key=key,
                Body=payload.encode("utf-8"), ContentType="application/json")
        else:
            os.makedirs(os.path.dirname(path), exist_ok=True)
            open(path, "w").write(payload)
    return s_path, m_path


Connection: project.spark.compatibility | Run start time: 2026-07-22 00:49:07.877167 | Run duration : 0:00:00.761959s.


### Run options: apply to every table

In [6]:
%%pyspark project.spark.compatibility
# run options: same for every table
from datetime import date

CHECKS_BASE   = "s3://enstream-lake-silver-dev/poc/dq/checks"
DQ_ROOT       = "s3://enstream-lake-silver-dev/poc/dq"   # results land under here
TABLE_PREFIX  = "trust_score_v1_poc_silver"

RUN_DATE      = date.today().isoformat()   # e.g. "2026-07-16"
FILTER_SQL    = None                       # e.g. "event_date = DATE'2024-08-01'"; None = whole table
WRITE_RESULTS = True                       # False = compute + inspect, no S3 write
SPLIT_SIZE_MB = None                       # e.g. 512 -> fewer, larger read splits = fewer
                                           # concurrent S3 readers. Set it if a scan dies with
                                           # "Timeout waiting for connection from pool"; takes
                                           # effect on the next read, no session restart needed.


Connection: project.spark.compatibility | Run start time: 2026-07-22 00:49:08.643745 | Run duration : 0:00:01.262580s.


### Pick the table: run ONE of these four

Each cell sets `TABLE_FQN` and `CHECKS_PATH` together from one `DATASET` name, so
they can't drift apart. That matters: the results path comes from the **yaml's**
`dataset`, so a mismatched pair would write this table's numbers under another
table's name.

These are plain assignments: the cell you ran last is the one that counts. If you
edit a cell, **re-run it**: the Spark session is remote, so editing the text
without executing leaves the old value bound.

In [54]:
%%pyspark project.spark.compatibility
DATASET     = "account_changes_batch"
TABLE_FQN   = f"{TABLE_PREFIX}.{DATASET}"
CHECKS_PATH = f"{CHECKS_BASE}/dataset={DATASET}/checks.yaml"
print("selected:", TABLE_FQN, "|", CHECKS_PATH)

selected: trust_score_v1_poc_silver.account_changes_batch | s3://enstream-lake-silver-dev/poc/dq/checks/dataset=account_changes_batch/checks.yaml

Connection: project.spark.compatibility | Run start time: 2026-07-22 02:01:00.317728 | Run duration : 0:00:02.695445s.


In [45]:
%%pyspark project.spark.compatibility
DATASET     = "audit_trail_services_3"
TABLE_FQN   = f"{TABLE_PREFIX}.{DATASET}"
CHECKS_PATH = f"{CHECKS_BASE}/dataset={DATASET}/checks.yaml"
print("selected:", TABLE_FQN, "|", CHECKS_PATH)

selected: trust_score_v1_poc_silver.audit_trail_services_3 | s3://enstream-lake-silver-dev/poc/dq/checks/dataset=audit_trail_services_3/checks.yaml

Connection: project.spark.compatibility | Run start time: 2026-07-22 01:53:59.719388 | Run duration : 0:00:04.453607s.


In [29]:
%%pyspark project.spark.compatibility
DATASET     = "device_lookup_batch"
TABLE_FQN   = f"{TABLE_PREFIX}.{DATASET}"
CHECKS_PATH = f"{CHECKS_BASE}/dataset={DATASET}/checks.yaml"
print("selected:", TABLE_FQN, "|", CHECKS_PATH)

selected: trust_score_v1_poc_silver.device_lookup_batch | s3://enstream-lake-silver-dev/poc/dq/checks/dataset=device_lookup_batch/checks.yaml

Connection: project.spark.compatibility | Run start time: 2026-07-22 01:20:00.465562 | Run duration : 0:00:03.165344s.


In [21]:
%%pyspark project.spark.compatibility
DATASET     = "tu_portps"
TABLE_FQN   = f"{TABLE_PREFIX}.{DATASET}"
CHECKS_PATH = f"{CHECKS_BASE}/dataset={DATASET}/checks.yaml"
print("selected:", TABLE_FQN, "|", CHECKS_PATH)

selected: trust_score_v1_poc_silver.tu_portps | s3://enstream-lake-silver-dev/poc/dq/checks/dataset=tu_portps/checks.yaml

Connection: project.spark.compatibility | Run start time: 2026-07-22 01:17:55.534826 | Run duration : 0:00:04.262705s.


### Run: read the silver table

In [55]:
%%pyspark project.spark.compatibility
# spark is already bound by the compatibility magic (same as the cleaning notebooks);
# spark.table reads the Iceberg/Glue table directly no special reader needed.
reader = spark.read
if SPLIT_SIZE_MB:
    # Iceberg ignores spark.sql.files.maxPartitionBytes; its split size is the
    # read.split.target-size table property, overridable per read with this option.
    reader = reader.option("split-size", str(SPLIT_SIZE_MB * 1024 * 1024))
df = reader.table(TABLE_FQN)
if FILTER_SQL:
    df = df.where(FILTER_SQL)

print(TABLE_FQN, "| columns:", len(df.columns))
print(df.columns)

# Stop here, not after the run: DQ on an empty table is meaningless every rule
# passes vacuously. take(1) is a LIMIT 1, so this costs nothing on a 400M-row table
# and doesn't add a pass the way count() would.
if not df.take(1):
    raise ValueError(
        f"{TABLE_FQN} returned no rows (FILTER_SQL={FILTER_SQL!r}). Publish the table "
        f"or fix the filter before running DQ; otherwise summary.json claims passed=true."
    )
print("not empty; ok to run DQ")

trust_score_v1_poc_silver.account_changes_batch | columns: 14
['record_id', 'phone_number_AC_hash', 'event_type', 'event_timestamp', 'mno', 'subId', 'msisdnChangeSide', 'correlationId', 'otherMSISDN', 'ingestion_ts', 'source_name', 'source_event_id', 'schema_version', 'event_date']
not empty; ok to run DQ

Connection: project.spark.compatibility | Run start time: 2026-07-22 02:01:03.016534 | Run duration : 0:00:05.194893s.


### Run: load the checks and compute both outputs

In [56]:
%%pyspark project.spark.compatibility
cfg = load_checks(CHECKS_PATH)

# The results path is built from the YAML's dataset, not from TABLE_FQN a mismatched
# pair would file this table's numbers under another table's name. Catch it here.
assert TABLE_FQN.split(".")[-1] == cfg.dataset, (
    f"CHECKS_PATH says dataset={cfg.dataset!r} but TABLE_FQN is {TABLE_FQN!r}. "
    f"Re-run one of the four table cells; don't edit them by hand."
)
summary, metrics = compute_dq(df, cfg, RUN_DATE)
print(f"{cfg.dataset}  run_date={RUN_DATE}")
print(f"rows={summary['row_count']:,}  rules={summary['rule_count']}  "
      f"passed={summary['passed_count']}  failed={summary['failed_count']}  "
      f"=> {'PASS' if summary['passed'] else 'FAIL'}")


account_changes_batch  run_date=2026-07-22
rows=351,823,492  rules=13  passed=13  failed=0  => PASS

Connection: project.spark.compatibility | Run start time: 2026-07-22 02:01:08.214705 | Run duration : 0:04:42.484649s.


### Inspect: read both files before writing them

Failing rules first as an index, then the complete `summary.json` and
`metrics.json` exactly as they'll be written.

In [57]:
%%pyspark project.spark.compatibility
# Quick index into the two dumps below: which rules failed, and the per-MNO shape.
fails = [r for r in summary["rules"] if not r["passed"]]
print(f"{summary['passed_count']}/{summary['rule_count']} rules passed"
      + ("" if not fails else f" ;  {summary['failed_count']} FAILED:"))
for r in fails:
    print(f"  [{r['family']:<12}] {r['rule']:<45} violations={r['violations']:,}")

print(f"\nper_mno groups: {list(metrics['per_mno']) or '(none; partition_by is null)'}")
for mno, blk in metrics["per_mno"].items():
    print(f"  {mno:<10} rows={blk['row_count']:,}")

13/13 rules passed

per_mno groups: ['TELUS', 'BELL', 'ROGERS']
  TELUS      rows=202,192,737
  BELL       rows=74,861,493
  ROGERS     rows=74,769,262
Connection: project.spark.compatibility | Run start time: 2026-07-22 02:05:50.702793 | Run duration : 0:00:01.258526s.


In [58]:
%%pyspark project.spark.compatibility
# FULL summary.json byte-for-byte what write_results puts in S3.
print(_dumps(summary))

{
  "dataset": "account_changes_batch",
  "failed_count": 0,
  "passed": true,
  "passed_count": 13,
  "row_count": 351823492,
  "rule_count": 13,
  "rules": [
    {
      "column": "event_timestamp",
      "family": "completeness",
      "passed": true,
      "rule": "event_timestamp is not null",
      "violations": 0
    },
    {
      "column": "event_type",
      "family": "completeness",
      "passed": true,
      "rule": "event_type is not null",
      "violations": 0
    },
    {
      "column": "mno",
      "family": "completeness",
      "passed": true,
      "rule": "mno is not null",
      "violations": 0
    },
    {
      "column": "phone_number_AC_hash",
      "family": "completeness",
      "passed": true,
      "rule": "phone_number_AC_hash is not null",
      "violations": 0
    },
    {
      "column": "record_id",
      "family": "completeness",
      "passed": true,
      "rule": "record_id is not null",
      "violations": 0
    },
    {
      "column": "schema_v

In [59]:
%%pyspark project.spark.compatibility
# FULL metrics.json byte-for-byte what write_results puts in S3.
# Long by design (every column's null rate, overall + per MNO). Read it before writing.
print(_dumps(metrics))

{
  "dataset": "account_changes_batch",
  "overall": {
    "completeness": {
      "per_column": {
        "correlationId": {
          "null_count": 0,
          "null_rate": 0.0
        },
        "event_date": {
          "null_count": 0,
          "null_rate": 0.0
        },
        "event_timestamp": {
          "null_count": 0,
          "null_rate": 0.0
        },
        "event_type": {
          "null_count": 0,
          "null_rate": 0.0
        },
        "ingestion_ts": {
          "null_count": 0,
          "null_rate": 0.0
        },
        "mno": {
          "null_count": 0,
          "null_rate": 0.0
        },
        "msisdnChangeSide": {
          "null_count": 321032131,
          "null_rate": 0.912481
        },
        "otherMSISDN": {
          "null_count": 328821310,
          "null_rate": 0.93462
        },
        "phone_number_AC_hash": {
          "null_count": 0,
          "null_rate": 0.0
        },
        "record_id": {
          "null_count": 0,
     

### Validate: outputs must match the committed JSON Schema

In [60]:
%%pyspark project.spark.compatibility
import jsonschema
SCHEMA_BASE = "s3://enstream-lake-silver-dev/poc/dq/schemas"

def validate(summary, metrics, schema_base):
    """Raise jsonschema.ValidationError if either output breaks the committed contract."""
    for obj, name in ((summary, "summary"), (metrics, "metrics")):
        bucket, key = f"{schema_base}/{name}.schema.json"[len("s3://"):].split("/", 1)
        jsonschema.validate(obj, json.loads(boto3.client("s3").get_object(Bucket=bucket, Key=key)["Body"].read()))

validate(summary, metrics, SCHEMA_BASE)
print("summary + metrics both validate against the committed schema")

summary + metrics both validate against the committed schema

Connection: project.spark.compatibility | Run start time: 2026-07-22 02:05:59.283398 | Run duration : 0:00:03.879366s.


### Write: canonical S3 paths

In [61]:
%%pyspark project.spark.compatibility
s_path, m_path = result_paths(DQ_ROOT, cfg, RUN_DATE)
if WRITE_RESULTS:
    write_results(summary, metrics, DQ_ROOT, cfg, RUN_DATE)
    print("WROTE", s_path)
    print("WROTE", m_path)
else:
    print("WRITE_RESULTS=False: would write to:")
    print(" ", s_path)
    print(" ", m_path)


('s3://enstream-lake-silver-dev/poc/dq/results/dataset=account_changes_batch/run_date=2026-07-22/summary.json', 's3://enstream-lake-silver-dev/poc/dq/results/dataset=account_changes_batch/run_date=2026-07-22/metrics.json')
WROTE s3://enstream-lake-silver-dev/poc/dq/results/dataset=account_changes_batch/run_date=2026-07-22/summary.json
WROTE s3://enstream-lake-silver-dev/poc/dq/results/dataset=account_changes_batch/run_date=2026-07-22/metrics.json
Connection: project.spark.compatibility | Run start time: 2026-07-22 02:06:03.165851 | Run duration : 0:00:03.361646s.


In [53]:
%%pyspark project.spark.compatibility



Connection: project.spark.compatibility | Run start time: 2026-07-22 02:00:46.815560 | Run duration : 0:00:00.036405s.


In [20]:
%%pyspark project.spark.compatibility



Connection: project.spark.compatibility | Run start time: 2026-07-22 01:03:17.919875 | Run duration : 0:00:00.045402s.


In [ ]:
%%pyspark project.spark.compatibility

